# Maya 2 voice server on a free Colab GPU
Makes Studio and live conversation speak in the cloned voice with Maya 2 (English, Hindi, Hinglish), first sound in about 0.13 s.

1. **Runtime → Change runtime type → T4 GPU**, Save.
2. Run the cells in order. The last cell prints two lines: paste them into `apps/web/.env.local` **and** `apps/voice/.env.local` on your Mac, then restart the web app and the voice service.

Keep this tab open while you use it; Colab stops it after a while of inactivity, and the link changes every time. Recordings of staff voices are sent here when a voice is first used: use your own Google account and only voices whose owners agreed. For testing, not for client calls.

In [ ]:
!nvidia-smi | head -15

In [ ]:
!pip -q install "maya3[server] @ git+https://github.com/MayaResearch/maya2-native" aiohttp

In [ ]:
import subprocess, time, urllib.request
server = subprocess.Popen(['python', '-m', 'maya3.server', '--ckpt', 'maya-research/maya2-native', '--host', '127.0.0.1', '--port', '8080', '--max-seats', '2'], stdout=open('/content/maya.log', 'w'), stderr=subprocess.STDOUT)
for i in range(180):
    try:
        if urllib.request.urlopen('http://127.0.0.1:8080/health', timeout=2).status == 200: print('Maya 2 ready'); break
    except Exception: time.sleep(5)
else:
    print(open('/content/maya.log').read()[-3000:])

The Maya 2 server has no login of its own, so this small gate only lets requests with the secret token through.

In [ ]:
import secrets, asyncio, threading
from aiohttp import web, ClientSession
TOKEN = secrets.token_urlsafe(32)
async def gate(request):
    if request.headers.get('authorization') != f'Bearer {TOKEN}':
        return web.Response(status=401, text='bad token')
    body = await request.read()
    async with ClientSession() as s:
        async with s.request(request.method, f'http://127.0.0.1:8080{request.path_qs}', data=body or None,
                             headers={k: v for k, v in request.headers.items() if k.lower() in ('content-type',)}) as r:
            resp = web.StreamResponse(status=r.status, headers={k: v for k, v in r.headers.items() if k.lower() in ('content-type', 'x-sample-rate', 'x-encoding')})
            await resp.prepare(request)
            async for chunk in r.content.iter_any(): await resp.write(chunk)
            await resp.write_eof(); return resp
async def start():
    app = web.Application(client_max_size=64 * 1024 * 1024); app.router.add_route('*', '/{tail:.*}', gate)
    runner = web.AppRunner(app); await runner.setup(); await web.TCPSite(runner, '127.0.0.1', 8081).start()
def run():
    loop = asyncio.new_event_loop(); loop.run_until_complete(start()); loop.run_forever()
threading.Thread(target=run, daemon=True).start(); time.sleep(2); print('gate on 8081')

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
import re
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8081'], stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)
url = None
for i in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m: url = m.group(0); break
    time.sleep(2)
print()
print('Paste these two lines into apps/web/.env.local AND apps/voice/.env.local:')
print()
print(f'MAYA2_URL={url}')
print(f'MAYA2_TOKEN={TOKEN}')